# Fig. 5a–c, Supplementary Fig. 7 and Supplementary Table 1

Order: simulation engine; each rule's retention-task cell (plateau search, Supp. Methods S1); full-fidelity runs (cached); learning/forgetting curves (Fig. 5a,b for PC, Supp. Fig. 7 for all rules); PC attractor clouds under each retention condition (Fig. 5c); statistics (Supp. Table 1).

In [ ]:
# ---- imports ----
from pathlib import Path
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ssb.utils import batch_inits
from ssb.scripts import run
from ssb.models import ssb, rules
from ssb.envs import bandits
from ssb.analysis.best_params import best_params


In [ ]:
# =====================================================================
#  CONFIG  —  all the knobs live here.  Edit, then Run All.
# =====================================================================
FIGDIR = Path('.')         # save next to this notebook (matches F1_demo/F2_SFig_magrate)
SAVE   = True              # save .svg next to the other paper figures (svg only, no png)

# ---------- Forgetting curves (Experiments 1 & 2, PC rule) ----------
SEED         = 11          # base random seed
N_SEEDS      = 250         # networks simulated in parallel  (↑ -> smoother median/IQR bands)
TRAIN_STEPS  = 100_000     # steps of training before retention  (also the LEARNING curve) --
RETAIN_STEPS = 100_000     # length of the retention (forgetting) phase -- both match the det
                           # sweep's own timescale (data/det/det_{rule}_{size}.npz is a 100k-step
                           # run), so 'best cell' selection and the actual figure use the same
                           # horizon. Used by section 4's rule-wide screen too (below).

# Retention conditions:  name -> reward delivered to (arm0, arm1) during retention
CONDITIONS = {
    'No input':           (0., 0.),    # no drive  -> plasticity gated OFF
    'Input, no new task': (1., 1.),    # input present, but no contingency
    'Reversal':           (1., 0.),    # the OTHER arm now pays -> a new task
}
COND_COLOR = {'No input': '#4FA06E', 'Input, no new task': '#5B7FB4', 'Reversal': '#C75D6E'}

# PC/COV hyperparameters: alpha=activity rate, beta=weight rate, eta=activity
# noise, tau=state timescale. Sourced from each rule's best deterministic-task
# operating point (ssb.analysis.best_params -- same function, same 'small' det
# sweep, as F1_demo.ipynb) rather than hand-tuned per-notebook constants.
_pc_eta,  _pc_alpha,  _pc_beta,  _pc_tau  = best_params('PC',  'small')
_cov_eta, _cov_alpha, _cov_beta, _cov_tau = best_params('COV', 'small')
HP_PC  = dict(alpha=_pc_alpha,  beta=_pc_beta,  eta=_pc_eta,  tau=_pc_tau)
HP_COV = dict(alpha=_cov_alpha, beta=_cov_beta, eta=_cov_eta, tau=_cov_tau)

# 'Custom' PC operating point: the ORIGINAL hand-picked values (not on the det-sweep
# grid). Why this exists alongside best_params' HP_PC: PC/small's det-sweep score is
# saturated at a perfect 1.0 across a wide span of eta (1e-5 through 1e-2 all score
# 1.0 -- see the 'Custom PC parameters' section below), so best_params has no signal
# telling it to prefer more noise and lands on the smallest allowed eta (1e-5). That
# happens to be exactly what mutes the input-gated forgetting story this figure is
# about: PC's plasticity is gated by the input unit's activity, but WHETHER an open
# gate actually erodes the old memory (under 'Input, no new task') depends on eta --
# with eta=1e-5 the network just sits at its trained fixed point since there's nothing
# to perturb it away; with more noise it drifts. So HP_PC_CUSTOM keeps a real, working
# eta for that comparison, matching what this notebook originally used.
HP_PC_CUSTOM = dict(alpha=.01, beta=.02, eta=.003, tau=0.)

# plotting knobs for the curves
N_PLOT_PTS = 400            # points drawn per curve (decimation; lower -> lighter SVG)
N_TRACE    = 14             # individual-seed traces drawn in the companion traces figure
DW_YLIM    = (-0.3, 0.40)  # y-range of the memory (Δw) panels (tight -> movements look dramatic)

# COV rule, plotted with the SAME stability figure (point attractor -> passive decay)
DW_YLIM_COV  = (-0.10, 0.18)

# ---------- Attractor geometry (Experiment 3, PC line attractor) ----------
ATTR_HP           = HP_PC_CUSTOM   # NOT best_params (HP_PC) -- best_params' eta=1e-5 is too small
                                   # for 'drive'/'reversal' to actually move anywhere within a sane
                                   # retention window (see 'Custom PC parameters' above for the same
                                   # issue in the forgetting curves). Costs a little cleanliness on
                                   # 'trained'/'no input': at this eta, training never fully empties
                                   # the negative branch (~9-11% of seeds sit there at ANY training
                                   # length, not just short ones -- it's ongoing noise re-crossing
                                   # zero, not slow convergence), vs. 0% under best_params. Kept
                                   # anyway since 'drive' spanning the line and 'reversal' actually
                                   # flipping sign are the more important things for this figure to show.
ATTR_TRAIN_STEPS  = 100_000        # PC's line attractor is bistable off a fresh init: seeds split
                                   # across a positive and negative branch. At best_params' eta this
                                   # is a slow saddle that fully unwinds by ~1e5 steps (which is why
                                   # this is 100k and not the original 8k); under HP_PC_CUSTOM's larger
                                   # noise a small residual fraction persists regardless (see above).
ATTR_RETAIN_STEPS = 20_000
ATTR_N_SEEDS      = 120
ATTR_COND = {                       # conditions shown as panels (reward pair during retention)
    'trained':  (0., 1.),           #   keep training -> the reference attractor
    'noinput':  (0., 0.),
    'drive':    (1., 1.),
    'reversal': (1., 0.),
}
ATTR_COLOR = {'trained': '#7991b5', 'noinput': '#ea8d50', 'drive': '#d76577', 'reversal': '#74ad83'}
ATTR_TITLE = {'trained': 'Trained', 'noinput': 'No input', 'drive': 'Drive (no task)', 'reversal': 'Reversal'}
ATTR_LIM   = ((-0.5, 0.5), (-0.5, 0.5))   # (xlim Δw, ylim Δx)

# example-trajectory knobs (a single, clearly-marked example per panel)
TRAJ_STEPS  = 10_000                # under HP_PC_CUSTOM's noisier dynamics, reversal doesn't
                                   # cross to the other side until several thousand steps in (see
                                   # CONFIG comment on ATTR_HP) -- too short and the example line
                                   # gets cut off before showing the actual flip
TRAJ_SEED   = 108                 # which seed's trajectory to draw as the example
TRAJ_STRIDE = 50                  # plot every Nth step of the trajectory (visual thinning; kept
                                   # roughly the same segment COUNT as before despite the longer span)
TRAJ_CMAP   = 'plasma'            # time colormap for the example (start=dark, end=bright)
CLOUD_PTS   = 1200                # points sampled for each settled-attractor cloud

# ---------- typography ----------
FS_S, FS_L = 6, 8                 # small / large font sizes (paper format)
plt.rcParams.update({'font.family': 'Arial', 'mathtext.fontset': 'stixsans', 'svg.fonttype': 'none',
                     'font.size': FS_S, 'axes.linewidth': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2,
                     'xtick.major.width': 0.5, 'ytick.major.width': 0.5})


## Simulation engine
One function does everything: train the small net, then run a retention phase logging the full state. Both the forgetting-curve and attractor sections call it; only the CONFIG values differ.

In [ ]:
import matplotlib.ticker as mticker

def _hps(hp, n_seeds, seed):
    """Build a batched hyperparameter dict for the small net (sizes [[1,0],[2,0]])."""
    h = batch_inits.make_default_hps(seed)
    h['alpha'] = jnp.array([hp['alpha']]); h['beta'] = jnp.array([hp['beta']])
    h['eta_a'] = jnp.array([hp['eta']]);   h['tau']   = jnp.array([hp['tau']])
    return batch_inits.batchify_hps(h, batch_size=n_seeds)

def train_net(rule, hp, train_steps, n_seeds, seed=SEED, log=False, log_actions=False):
    """Train n_seeds nets on the 2AFC task (arm 1 rewarded). Returns (hps_batch, final_state).
    If log=True, also return the LEARNING curve dw_train (train_steps, n_seeds) = w1-w2 over
    training. If BOTH log=True and log_actions=True, also return action_train (train_steps,
    n_seeds) -- run.run_env's log already contains the action at index 0 regardless of
    log_weights; log_actions just controls whether train_net extracts and returns it, so
    existing 2-/3-tuple callers (cells above) are unaffected."""
    hb = _hps(hp, n_seeds, seed)
    logd, fs = run.run_env(hb, rule, bandits.bandit_det, jnp.array([0., 1.]),
                           timesteps=train_steps, log_weights=log)
    if log:
        w = np.asarray(logd[3]); dw_train = w[:, :, 0, 2] - w[:, :, 0, 1]   # (T, n_seeds)
        if log_actions:
            return hb, fs, dw_train, np.asarray(logd[0])   # (T, n_seeds), arm chosen each step
        return hb, fs, dw_train
    return hb, fs

def retain_full(rule, hb, fs, retain_par, retain_steps):
    """Run the retention phase from a trained checkpoint, logging the full state every step.
    Returns array (retain_steps, n_seeds, 6) with columns:  x0, x1, x2, w01, w02, action
    (action = arm chosen that step, 0 or 1 -- see the action() accessor below)."""
    uw, us = rules.rule2fns(rule)
    ua = jax.vmap(ssb.update_acts); uw_ = jax.vmap(uw); us_ = jax.vmap(us)
    noise = jax.vmap(ssb.act_noise); envb = jax.vmap(bandits.bandit_det); expand = jax.vmap(ssb.expand_stimulus)
    acts, states, weights, tern, key = fs[1], fs[2], fs[3], fs[4], fs[5]
    n = acts.shape[0]
    ep = jnp.tile(jnp.array(list(retain_par))[None], (n, 1))   # reward pair for this condition
    @jax.jit
    def step(c, _):
        acts, states, weights, key = c
        r, act, _, key = envb(jnp.zeros((n, 2)), acts[:, -2:], ep, key)
        stim = expand(r, acts)
        acts = ua(stim, acts, states, weights, tern, hb)
        acts, key = noise(acts, key, hb)
        states = us_(acts, states, hb)
        weights = uw_(stim, acts, states, weights, tern, hb)
        return (acts, states, weights, key), jnp.concatenate(
            [acts[:, :3], weights[:, 0, 1:3], act[:, None].astype(jnp.float32)], axis=1)
    _, out = jax.lax.scan(step, (acts, states, weights, key), None, length=retain_steps)
    return np.asarray(out)

def run_forgetting(rule, retain_par, hp, train_steps, retain_steps, n_seeds, seed=SEED):
    """Train, then run one retention condition. -> state array (retain_steps, n_seeds, 6)."""
    hb, fs = train_net(rule, hp, train_steps, n_seeds, seed)
    return retain_full(rule, hb, fs, retain_par, retain_steps)

# --- readouts from a state array ---
def w_opt(state):   return state[:, :, 4]                    # weight to the originally-rewarded arm
def w_sub(state):   return state[:, :, 3]                    # weight to the other arm
def memory(state):  return state[:, :, 4] - state[:, :, 3]   # Δw  (the stored memory)
def readout(state): return state[:, :, 2] - state[:, :, 1]   # Δx  (activity readout of the memory)
def action(state):  return state[:, :, 5]                    # arm chosen that step (0/1, retain_full only)

# --- small plotting helpers (Nature style, matching F3_pm.ipynb) ---
def decimate(n, npts):
    """Indices thinning n points to ~npts, denser early (looks right on linear AND log axes)."""
    return np.unique(np.geomspace(1, n, npts).astype(int)) - 1

def band(ax, t, X, color, idx, lw=1.3, label=None, alpha=0.18):
    """Median line + interquartile (25–75%) shaded band, decimated to idx."""
    med = np.median(X, 1); p25 = np.percentile(X, 25, 1); p75 = np.percentile(X, 75, 1)
    ax.fill_between(t[idx], p25[idx], p75[idx], color=color, alpha=alpha, lw=0, zorder=2)
    ax.plot(t[idx], med[idx], color=color, lw=lw, label=label, zorder=4)

def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

def nature_ax(ax, xlabel='', ylabel='', nticks=4, nx=None, ny=None):
    """F3_pm styling: hide top/right spines, thin axes/ticks, few ticks, 7–9 pt labels."""
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx or nticks))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(ny or nticks))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

def _smooth(y, w):
    """Light centred moving-average for display only (edge-safe)."""
    w = int(max(3, w))
    return np.convolve(np.pad(y, w // 2, mode='edge'), np.ones(w) / w, mode='valid')[:len(y)]

def mean_sd(ax, t, X, color, idx, lw_mean=1.7, sd_alpha=0.20, smooth_w=0):
    """Bold (lightly-smoothed) MEAN with a colored ±SD shaded band — no individual traces."""
    m = X.mean(1); sd = X.std(1)
    if smooth_w: m = _smooth(m, smooth_w); sd = _smooth(sd, smooth_w)
    ax.fill_between(t[idx], (m - sd)[idx], (m + sd)[idx], color=color, alpha=sd_alpha, lw=0, zorder=3)
    ax.plot(t[idx], m[idx], color=color, lw=lw_mean, zorder=4)

def draw_traces(ax, t, X, color, idx, n_show=N_TRACE, alpha=0.5, lw=0.1, smooth_w=0):
    """A sample of individual-seed traces (lightly smoothed, in colour) + a thin mean on top."""
    S = X.shape[1]; sel = np.linspace(0, S - 1, min(n_show, S)).astype(int)
    segs = [np.column_stack([t[idx], (_smooth(X[:, s], smooth_w) if smooth_w else X[:, s])[idx]])
            for s in sel]
    ax.add_collection(LineCollection(segs, colors=[color], linewidths=lw, alpha=alpha,
                                     rasterized=False, zorder=2))
    m = X.mean(1)
    if smooth_w: m = _smooth(m, smooth_w)
    ax.plot(t[idx], m[idx], color=color, lw=1.0, zorder=4)

def plot_forgetting(forget, dw_train, dw_ylim, suptitle, fname):
    """Single-row stability figure (F3_pm style): the LEARNING curve (Δw acquired during
    training) on the left, then one memory-Δw retention panel per condition. Shared y-axis,
    LINEAR time (×10³ steps). Error is a colored ±SD shaded band behind a lightly-smoothed
    bold mean (individual seeds live in the companion traces figure). Reused for PC & COV."""
    names = list(forget)
    Tt = dw_train.shape[0]; tt = np.arange(Tt) / 1e3; idt = decimate(Tt, N_PLOT_PTS)
    P  = forget[names[0]].shape[0]; tr = np.arange(P) / 1e3; idx = decimate(P, N_PLOT_PTS)
    trained = float(np.mean(dw_train[-1]))

    fig, axes = plt.subplots(1, 1 + len(names), figsize=(1.7 * (1 + len(names)), 1.4), sharey=True,
                             gridspec_kw=dict(width_ratios=[0.9] + [1] * len(names), wspace=0.18))
    # --- learning curve: Δw rises from 0 to the trained level over training ---
    axL = axes[0]
    axL.axhline(0, color='#999999', lw=0.5, zorder=1)
    mean_sd(axL, tt, dw_train, '#8c6ba8', idt, smooth_w=max(3, Tt // 120))
    axL.axhline(trained, color='#888888', lw=0.5, zorder=1)
    axL.set_xlim(0, tt[-1])
    nature_ax(axL, 'training ($\\times10^3$)', 'memory  $\\Delta w$', nx=3, ny=4)
    axL.set_title('Learning', fontsize=FS_L, color='#8c6ba8'); axL.set_ylim(*dw_ylim)
    # --- retention panels: each starts at the trained level and stays / decays / reverses ---
    for j, name in enumerate(names):
        ax = axes[j + 1]; st = forget[name]
        ax.axhline(0, color='#999999', lw=0.5, zorder=1)
        ax.axhline(trained, color='#cccccc', lw=0.5, zorder=1)
        mean_sd(ax, tr, memory(st), COND_COLOR[name], idx, smooth_w=max(3, P // 120))
        ax.set_xlim(0, tr[-1])
        nature_ax(ax, 'retention ($\\times10^3$)' if j == 1 else '', nx=3, ny=4)
        ax.set_title(name, fontsize=FS_L, color=COND_COLOR[name]); ax.set_ylim(*dw_ylim)
    fig.tight_layout()
    if SAVE:
        fig.savefig(FIGDIR / (fname + '.svg'), bbox_inches='tight')
    plt.show()


def plot_traces(forget, dw_train, dw_ylim, fname):
    """Companion (shorter) figure: the individual-seed traces behind each mean — same panels,
    colours and axes as the mean±SD figure, just smaller. Pairs with plot_forgetting."""
    names = list(forget)
    Tt = dw_train.shape[0]; tt = np.arange(Tt) / 1e3; idt = decimate(Tt, N_PLOT_PTS)
    P  = forget[names[0]].shape[0]; tr = np.arange(P) / 1e3; idx = decimate(P, N_PLOT_PTS)
    trained = float(dw_train[-1].mean())
    fig, axes = plt.subplots(1, 1 + len(names), figsize=(1.7 * (1 + len(names)), 1.05), sharey=True,
                             gridspec_kw=dict(width_ratios=[0.9] + [1] * len(names), wspace=0.18))
    axL = axes[0]
    axL.axhline(0, color='#999999', lw=0.5, zorder=1)
    draw_traces(axL, tt, dw_train, '#8c6ba8', idt, smooth_w=max(3, Tt // 120))
    axL.axhline(trained, color='#cccccc', lw=0.5, zorder=1)
    axL.set_xlim(0, tt[-1]); nature_ax(axL, 'training ($\\times10^3$)', 'memory  $\\Delta w$', nx=3, ny=4)
    axL.set_title('Learning', fontsize=FS_S, color='#8c6ba8'); axL.set_ylim(*dw_ylim)
    for j, name in enumerate(names):
        ax = axes[j + 1]; st = forget[name]
        ax.axhline(0, color='#999999', lw=0.5, zorder=1)
        ax.axhline(trained, color='#cccccc', lw=0.5, zorder=1)
        draw_traces(ax, tr, memory(st), COND_COLOR[name], idx, smooth_w=max(3, P // 120))
        ax.set_xlim(0, tr[-1]); nature_ax(ax, 'retention ($\\times10^3$)' if j == 1 else '', nx=3, ny=4)
        ax.set_title(name, fontsize=FS_S, color=COND_COLOR[name]); ax.set_ylim(*dw_ylim)
    fig.tight_layout()
    if SAVE:
        fig.savefig(FIGDIR / (fname + '.svg'), bbox_inches='tight')
    plt.show()


## 4. Rule-wide check: does every rule's best-performing cell show the dissociation?

`best_params` breaks ties on raw task performance alone. For PC/small that landed on `eta=1e-5`
-- not because more noise hurts performance (the det-sweep score is a saturated **1.0** across a
wide span of eta), but because argmax has no reason to prefer noise when it doesn't move the
score -- and that happened to disable the stable/decays dissociation itself (see "Custom PC
parameters" above).

So here every rule gets the same check. For any rule whose det-sweep score has a real *tied*
plateau at its maximum (more than a cell or two), we search that whole plateau for the cell with
the strongest retention contrast -- `retention(No input) - retention(Input, no new task)` -- after
first throwing out cells whose retention ratio blows up outside a sane range (the same large-alpha
instability that showed up with the earlier magrate-derived candidates). Rules whose max score is
**not** a plateau (OJA, BCM, LC -- only 1-2 cells reach their own, imperfect best) have no tie to
search, so `best_params` already picks essentially the only good cell there.

The screen uses 20 seeds/cell (cheap, for ranking only, and keeps only the final retention ratio
-- no per-step trajectory -- so it can scan thousands of lanes at once); the table and plots below
then re-simulate each rule's winning cell at this notebook's usual fidelity (`N_SEEDS`,
`RETAIN_STEPS`), so the reported numbers match what's plotted.


In [ ]:
import os
from ssb.analysis.cell_select import ALPHAS, BETAS, ETAS, TAUS, RULES_WITH_TAU
from ssb.analysis.best_params import DET_FOLDER

CACHE_DIR = Path('cache')   # screen_plateau results cached here -- the 896-cell
                            # plateau search is the expensive part of section 4
                            # (tens of minutes); the final per-rule confirmation
                            # below is cheap enough to just always rerun fresh.

PLATEAU_RULES = ['PC', 'HR', 'COV', 'HZ']       # real ties at the max det score
UNIQUE_RULES  = ['OJA', 'BCM', 'LC']            # <=2 cells at their own (imperfect) max
N_SCREEN_SEEDS = 20
MAX_LANES_PER_CHUNK = 20_000    # generous -- see train_final_only/retain_final_only below
SCREEN_STABLE_LO, SCREEN_STABLE_HI = -0.3, 1.3   # sane range for a retention ratio

def _det_score(rule, size='small'):
    x = np.load(DET_FOLDER / f'det_{rule}_{size}.npz')['x']
    mean_seeds = x.mean(axis=-3)
    last20k = mean_seeds[..., -20000:, :, :]
    score = np.median(last20k, axis=-3)
    score[:1] = -1                               # exclude eta_idx 0, same as best_params
    return score

def _max_score_cells(rule, size='small'):
    score = _det_score(rule, size)
    mx = score.max()
    idxs = np.argwhere(score >= mx - 1e-9)
    has_tau = rule in RULES_WITH_TAU
    cells = []
    for row in idxs:
        if has_tau:
            e, t, a, b = row; tau = float(TAUS[t])
        else:
            e, a, b = row; tau = 0.
        cells.append((float(ETAS[e]), float(ALPHAS[a]), float(BETAS[b]), tau))
    return cells, float(mx)

def _screen_hb(cells, n_seeds, seed0):
    n_lanes = len(cells) * n_seeds
    alpha = np.repeat([c[1] for c in cells], n_seeds)
    beta  = np.repeat([c[2] for c in cells], n_seeds)
    eta   = np.repeat([c[0] for c in cells], n_seeds)
    tau   = np.repeat([c[3] for c in cells], n_seeds)
    seeds = seed0 + np.arange(n_lanes)
    return {
        'sizes':      jnp.tile(jnp.array([[1, 0], [2, 0]]), (n_lanes, 1, 1)),
        'init_scale': jnp.ones((n_lanes, 1)), 'randinit': jnp.ones((n_lanes, 1)),
        'posinit':    jnp.ones((n_lanes, 1)), 'seed': jnp.array(seeds).reshape(-1, 1),
        'alpha':      jnp.array(alpha).reshape(-1, 1), 'beta': jnp.array(beta).reshape(-1, 1),
        'eta_a':      jnp.array(eta).reshape(-1, 1), 'cap': jnp.ones((n_lanes, 1)),
        'rho':        jnp.full((n_lanes, 1), 5.0), 'tau': jnp.array(tau).reshape(-1, 1),
    }

def train_final_only(rule, hb, timesteps):
    """Screening-only sibling of run.run_env: same training dynamics, but
    doesn't stack a per-step log (run_env does, even with log_weights=False --
    action/reward/last-act every step -- which is the actual memory driver
    behind chunking, not lane count itself). Only the final state is ever
    read here, so there's nothing to gain from materializing the trajectory."""
    update_weights, update_states = rules.rule2fns(rule)
    update_acts_batch = jax.vmap(ssb.update_acts); update_weights_batch = jax.vmap(update_weights)
    act_noise_batch = jax.vmap(ssb.act_noise); update_states_batch = jax.vmap(update_states)
    env_batch = jax.vmap(bandits.bandit_det); expand_batch = jax.vmap(ssb.expand_stimulus)
    n = hb['seed'].shape[0]
    env_par = jnp.tile(jnp.array([0., 1.])[None], (n, 1))
    acts, states, weights, tern, key = batch_inits.init_params_batch(hb)
    env_state = jnp.zeros((n, 2))
    @jax.jit
    def step(c, _):
        env_state, acts, states, weights, key = c
        reward, action, env_state, key = env_batch(env_state, acts[:, -2:], env_par, key)
        stim = expand_batch(reward, acts)
        acts = update_acts_batch(stim, acts, states, weights, tern, hb)
        acts, key = act_noise_batch(acts, key, hb)
        weights = update_weights_batch(stim, acts, states, weights, tern, hb)
        states = update_states_batch(acts, states, hb)
        return (env_state, acts, states, weights, key), None
    (env_state, acts, states, weights, key), _ = jax.lax.scan(
        step, (env_state, acts, states, weights, key), None, length=timesteps)
    return (None, acts, states, weights, tern, key)   # matches run_env's final_state indexing


def retain_final_only(rule, hb, fs, retain_par, retain_steps):
    """Screening-only sibling of retain_full: same dynamics, but keeps just the
    final weights (no per-step trajectory) -- light enough to scan thousands of
    (cell x seed) lanes in one batched call."""
    uw, us = rules.rule2fns(rule)
    ua = jax.vmap(ssb.update_acts); uw_ = jax.vmap(uw); us_ = jax.vmap(us)
    noise = jax.vmap(ssb.act_noise); envb = jax.vmap(bandits.bandit_det); expand = jax.vmap(ssb.expand_stimulus)
    acts, states, weights, tern, key = fs[1], fs[2], fs[3], fs[4], fs[5]
    n = acts.shape[0]
    ep = jnp.tile(jnp.array(list(retain_par))[None], (n, 1))
    @jax.jit
    def step(c, _):
        acts, states, weights, key = c
        r, _, _, key = envb(jnp.zeros((n, 2)), acts[:, -2:], ep, key)
        stim = expand(r, acts)
        acts = ua(stim, acts, states, weights, tern, hb)
        acts, key = noise(acts, key, hb)
        states = us_(acts, states, hb)
        weights = uw_(stim, acts, states, weights, tern, hb)
        return (acts, states, weights, key), None
    (acts, states, weights, key), _ = jax.lax.scan(step, (acts, states, weights, key), None, length=retain_steps)
    return np.asarray(weights[:, 0, 1:3])   # (n_lanes, 2): [w_sub, w_opt]

def _screen_cache_path(rule):
    return CACHE_DIR / f'screen_{rule}.npz'


def _screen_cache_matches(cache, n_cells, seed0):
    return (int(cache['n_cells']) == n_cells and int(cache['seed0']) == seed0 and
           int(cache['n_screen_seeds']) == N_SCREEN_SEEDS and
           int(cache['train_steps']) == TRAIN_STEPS and int(cache['retain_steps']) == RETAIN_STEPS)


def screen_plateau(rule, seed0=20_000, use_cache=True):
    """Search rule/small's tied-for-best det-sweep plateau for the cell with the
    strongest, stable retention contrast. Prints the plateau size as it goes.

    Caches per-cell results (NOT just the winner -- the raw per-lane trained_dw
    and per-condition final Δw too) to cache/screen_{rule}.npz, keyed on
    (n_cells, seed0, N_SCREEN_SEEDS, TRAIN_STEPS, RETAIN_STEPS). A parameter
    mismatch (e.g. TRAIN_STEPS changed) triggers a fresh sweep. Keeping the raw
    per-lane numbers, not just the aggregate ret_noinput/ret_notask, means a
    future change to the ranking statistic (like the mean-of-ratios ->
    ratio-of-means fix earlier) can be re-derived from the cache alone."""
    cells, mx = _max_score_cells(rule)
    n_cells = len(cells)

    cache_path = _screen_cache_path(rule)
    if use_cache and cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        if _screen_cache_matches(cached, n_cells, seed0):
            print(f'{rule}: loaded {n_cells}-cell plateau screen from cache ({cache_path})')
            ret_noinput, ret_notask = cached['ret_noinput'], cached['ret_notask']
            cells = [tuple(row) for row in cached['cells']]
        else:
            print(f'{rule}: cache at {cache_path} has different params -- rescreening')
            ret_noinput = ret_notask = None
    else:
        ret_noinput = ret_notask = None

    if ret_noinput is None:
        n_lanes = n_cells * N_SCREEN_SEEDS
        print(f'{rule}: sweeping {n_cells} cells tied at max det score ({mx:.4f}) '
             f'x {N_SCREEN_SEEDS} seeds = {n_lanes} lanes')

        n_chunks = max(1, -(-n_lanes // MAX_LANES_PER_CHUNK))
        chunk_size = -(-n_cells // n_chunks)
        ret_noinput_parts, ret_notask_parts = [], []
        for ci in range(n_chunks):
            sub = cells[ci * chunk_size:(ci + 1) * chunk_size]
            if not sub:
                continue
            hb = _screen_hb(sub, N_SCREEN_SEEDS, seed0)
            fs = train_final_only(rule, hb, TRAIN_STEPS)
            trained_dw = np.asarray(fs[3])[:, 0, 2] - np.asarray(fs[3])[:, 0, 1]
            w_noinput = retain_final_only(rule, hb, fs, (0., 0.), RETAIN_STEPS)
            w_notask  = retain_final_only(rule, hb, fs, (1., 1.), RETAIN_STEPS)
            dw_noinput = (w_noinput[:, 1] - w_noinput[:, 0]).reshape(len(sub), N_SCREEN_SEEDS)
            dw_notask  = (w_notask[:, 1]  - w_notask[:, 0]).reshape(len(sub), N_SCREEN_SEEDS)
            trained_2d = trained_dw.reshape(len(sub), N_SCREEN_SEEDS)
            # ratio of MEANS (population-level), not mean of per-seed ratios -- a
            # single seed whose own trained_dw lands near zero would otherwise blow
            # up that one ratio and can dominate the per-cell average.
            with np.errstate(divide='ignore', invalid='ignore'):
                ret_noinput_parts.append(dw_noinput.mean(axis=1) / trained_2d.mean(axis=1))
                ret_notask_parts.append(dw_notask.mean(axis=1)   / trained_2d.mean(axis=1))
            if n_chunks > 1:
                print(f'  {rule}: chunk {ci + 1}/{n_chunks} ({len(sub)} cells) done')
        ret_noinput = np.concatenate(ret_noinput_parts)
        ret_notask  = np.concatenate(ret_notask_parts)

        if use_cache:
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(cache_path, cells=np.array(cells), ret_noinput=ret_noinput,
                                ret_notask=ret_notask, n_cells=n_cells, seed0=seed0,
                                n_screen_seeds=N_SCREEN_SEEDS, train_steps=TRAIN_STEPS,
                                retain_steps=RETAIN_STEPS)
            print(f'  {rule}: cached to {cache_path}')

    stable = ((ret_noinput > SCREEN_STABLE_LO) & (ret_noinput < SCREEN_STABLE_HI) &
             (ret_notask  > SCREEN_STABLE_LO) & (ret_notask  < SCREEN_STABLE_HI))
    n_stable = int(stable.sum())
    print(f'  {rule}: {n_stable}/{n_cells} cells stay in a sane retention range; '
         f'picking the highest-contrast one')
    stable_idx = np.where(stable)[0]
    contrast = ret_noinput - ret_notask
    best_i = stable_idx[np.argmax(contrast[stable_idx])]
    eta, alpha, beta, tau = cells[best_i]
    return dict(eta=eta, alpha=alpha, beta=beta, tau=tau, n_tied=n_cells,
               method=f'plateau search (n={n_cells})')


In [ ]:
# ---- run the picks: plateau search where there's a real tie, best_params otherwise ----
PICKS = {}
for rule in PLATEAU_RULES:
    PICKS[rule] = screen_plateau(rule)
# HZ's max-contrast pick turned out to be a genuinely unstable outlier at full
# fidelity (ret_notask blew up to -111 .. -588 depending on the tail window --
# see section 5). A much cleaner cell sits in the SAME 370-cell plateau (found
# by scoring cache/screen_HZ.npz for closeness to the ideal retain=1/decay=0
# corner instead of raw contrast, which rewards extreme/unstable swings) --
# ret_noinput=0.988, ret_notask=-0.012 at the 20-seed screening level. Override
# the pick here; run_and_cache_final below detects the param mismatch against
# the stale cache/final_HZ.npz and reruns just this one rule.
PICKS['HZ'] = dict(eta=0.01, alpha=0.4394, beta=0.007197, tau=0.004642,
                   n_tied=PICKS['HZ']['n_tied'], method='plateau search, closest-to-clean (n=370)')

for rule in UNIQUE_RULES:
    eta, alpha, beta, tau = best_params(rule, 'small')
    score = _det_score(rule)
    n_tied = int((score >= score.max() - 1e-9).sum())
    print(f'{rule}: no real plateau (max det score {score.max():.4f}, {n_tied} cell(s) there) '
         f'-- using best_params directly')
    PICKS[rule] = dict(eta=float(eta), alpha=float(alpha), beta=float(beta), tau=float(tau),
                       n_tied=n_tied, method='best_params (no plateau)')


In [ ]:
# ---- run (or load from cache) each rule's picked cell at full fidelity --
#      TRAIN_STEPS/RETAIN_STEPS, N_SEEDS -- keeping the FULL-resolution Δw
#      AND action trajectory for every seed, every condition. No metric
#      decision happens in this cell: it only runs/caches the raw data, so
#      the next section can try different retention-ratio definitions
#      (last step? tail window? action-based P(chosen)?) without
#      resimulating anything. Cached to cache/final_{rule}.npz.
ALL_RULES = ['BCM', 'OJA', 'LC', 'PC', 'COV', 'HZ', 'HR']   # plotting order (requested)

def _final_cache_path(rule):
    return CACHE_DIR / f'final_{rule}.npz'


def _final_cache_matches(cache, p):
    return (int(cache['train_steps']) == TRAIN_STEPS and int(cache['retain_steps']) == RETAIN_STEPS and
           int(cache['n_seeds']) == N_SEEDS and int(cache['seed']) == SEED and
           float(cache['eta']) == p['eta'] and float(cache['alpha']) == p['alpha'] and
           float(cache['beta']) == p['beta'] and float(cache['tau']) == p['tau'])


def run_and_cache_final(rule, use_cache=True):
    p = PICKS[rule]
    cache_path = _final_cache_path(rule)
    if use_cache and cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        if _final_cache_matches(cached, p):
            print(f'{rule}: loaded full-fidelity run from cache ({cache_path})')
            dw_train = cached['dw_train']; action_train = cached['action_train']
            forget = {name: cached[f'state_{name}'] for name in CONDITIONS}
            return dw_train, action_train, forget
        print(f'{rule}: cache at {cache_path} has different params -- rerunning')

    hp = dict(alpha=p['alpha'], beta=p['beta'], eta=p['eta'], tau=p['tau'])
    hb, fs, dw_train, action_train = train_net(rule, hp, TRAIN_STEPS, N_SEEDS, log=True, log_actions=True)
    forget = {name: retain_full(rule, hb, fs, par, RETAIN_STEPS) for name, par in CONDITIONS.items()}

    if use_cache:
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        payload = dict(dw_train=dw_train, action_train=action_train,
                      train_steps=TRAIN_STEPS, retain_steps=RETAIN_STEPS, n_seeds=N_SEEDS, seed=SEED,
                      eta=p['eta'], alpha=p['alpha'], beta=p['beta'], tau=p['tau'])
        for name, arr in forget.items():
            payload[f'state_{name}'] = arr   # (retain_steps, n_seeds, 6): x0,x1,x2,w01,w02,action
        np.savez_compressed(cache_path, **payload)
        size_mb = cache_path.stat().st_size / 1e6
        print(f'{rule}: cached to {cache_path} ({size_mb:.1f} MB)')
    return dw_train, action_train, forget


dw_train_by_rule, action_train_by_rule, forgets_by_rule = {}, {}, {}
for rule in ALL_RULES:
    dw_train, action_train, forget = run_and_cache_final(rule)
    dw_train_by_rule[rule] = dw_train
    action_train_by_rule[rule] = action_train
    forgets_by_rule[rule] = forget
    trained = float(dw_train[-1].mean())
    p_optimal_end_train = float(action_train[-1].mean())   # P(choose arm 1) at end of training
    print(f"{rule}: trained Δw={trained:+.4f}  P(optimal) at end of training={p_optimal_end_train:.3f}")


## 6. Best retention curves per rule, in the original Learning/forgetting style

Same figure style as PC/COV above (`plot_forgetting` + `plot_traces`): Learning on the left, then
one panel per retention condition. Uses each rule's finalized pick from section 4 -- with HZ's
overridden to the clean cell found in section 5's reanalysis, since its raw max-contrast pick
turned out to be an unstable outlier, not a real dissociation. PC, HR and COV's picks were already
the best available in their own plateau (checked above); OJA, BCM and LC have no plateau to choose
from at all, so what's plotted for them is simply what `best_params` gives -- for BCM and LC in
particular, don't expect a clean stable/decays story, since neither ever showed one at any window
size in section 5.

y-limits are set per rule from that rule's own data (1st-99th percentile with padding), since
magnitudes vary by orders of magnitude across rules (PC ~0.2 vs. LC ~12). The mean+/-SD figure
(`plot_forgetting`) and the individual-traces figure (`plot_traces`) get DIFFERENT y-limits: the
traces figure is scoped to the raw per-seed range (it needs the room), while the mean+/-SD figure
is scoped to the MEAN LINES' own range -- a few noisy individual seeds (or, for BCM/LC/HZ, large
per-timestep SD) would otherwise blow even a band-based range out far past where the means
themselves ever go, flattening the No-input vs. Input-no-task difference this figure exists to
show. The SD shading is left free to clip at the plot edges for the unstable rules -- that clipping
is itself informative (it shows how much the band overshoots the mean-level story).


In [ ]:
def _auto_ylim(forget, dw_train, pad_frac=0.15):
    """Range of the RAW per-seed traces (1st-99th pctl) -- what plot_traces actually draws."""
    vals = np.concatenate([dw_train.ravel()] + [memory(forget[n]).ravel() for n in CONDITIONS])
    lo, hi = np.percentile(vals, [1, 99])
    pad = (hi - lo) * pad_frac or 0.05
    return (min(0., lo - pad), max(0., hi + pad))


def _auto_ylim_mean(forget, dw_train, pad_frac=0.15, pctl=(1, 99)):
    """Range of the MEAN LINES themselves (1st-99th pctl over time), not the +/-SD band --
    what the eye actually tracks in plot_forgetting. Scoping to the +/-SD band (tried first)
    didn't work: for the unstable rules (BCM, LC, HZ) a handful of seeds carry huge
    per-timestep SD, which blows the band out just as wide as the raw per-seed range and
    flattens the No-input/Input-no-task difference right back to nothing. The mean line's
    OWN range is much tighter and is exactly the comparison this figure is meant to show;
    the SD shading is allowed to clip at the edges (still informative -- it shows how far
    the band overshoots the mean-level story, which for BCM/LC is itself the point)."""
    def mean_line(X):
        return X.mean(1)
    vals = np.concatenate([mean_line(dw_train)] + [mean_line(memory(forget[n])) for n in CONDITIONS])
    lo, hi = np.percentile(vals, pctl)
    pad = (hi - lo) * pad_frac or 0.05
    return (min(0., lo - pad), max(0., hi + pad))


for rule in ALL_RULES:
    dw_train = dw_train_by_rule[rule]
    forget = forgets_by_rule[rule]
    ylim_mean = _auto_ylim_mean(forget, dw_train)
    ylim_traces = _auto_ylim(forget, dw_train)
    p = PICKS[rule]
    print(f"{rule}: eta={p['eta']:.4g} alpha={p['alpha']:.4g} beta={p['beta']:.4g} tau={p['tau']:.4g}  "
         f"({p['method']})  ylim_mean={tuple(round(v, 3) for v in ylim_mean)}  "
         f"ylim_traces={tuple(round(v, 3) for v in ylim_traces)}")
    plot_forgetting(forget, dw_train, ylim_mean,
                    f'{rule}: best available retention curves ({p["method"]})',
                    f'fig_bestcurve_{rule.lower()}')
    plot_traces(forget, dw_train, ylim_traces, f'fig_bestcurve_{rule.lower()}_traces')


### PC line attractor — non-custom parameters (plateau-search pick)

Same panels as above, but for the PC rule's **non-custom** parameters instead of the hand-picked
`HP_PC_CUSTOM`. Literal `best_params('PC', 'small')` alone lands on `eta=1e-5` (the smallest eta
the det sweep allows) — and at that eta neither `drive` nor `reversal` moves at all: checked by
directly simulating this cell out to 1,000,000 retention steps, `drive`'s Δw never leaves ~1-5% of
its trained value and `reversal` only ever creeps to about -0.001 to -0.004, regardless of run
length or seed. That's the same eta-degeneracy documented in "Custom PC parameters" above (PC/small's
det-sweep score is a saturated 1.0 across eta 1e-5–1e-2, so `best_params`' argmax has no reason to
prefer more noise) — `best_params` just happens to keep the *smallest* eta on that tied plateau,
while section 4's plateau search below (which breaks the tie by retention *contrast* instead of raw
score) finds a cell in the SAME plateau with `eta=0.01` (1000x more activity noise) that shows the
intended dissociation cleanly. That's `PICKS['PC']`, cached in `cache/final_PC.npz` and already
what section 6 plots for PC — so this is the notebook's own non-hand-picked "PC that actually
forgets," not a new tuned constant.

`TRAJ_SEED_PICK` below (one of `ATTR_N_SEEDS` seeds off the same trained checkpoint) was found by
scanning all seeds for one where, within the SAME `ATTR_RETAIN_STEPS` window used above: `no input`
never dips below Δw=0 (a tight rotating loop, Δw∈[0.244, 0.262], Δx swinging ±0.1-0.2 around it);
`drive` genuinely crosses the whole plotted diagonal (Δw: +0.26 → -0.30); `reversal` properly
relearns to a clear negative value (-0.26, min -0.35) rather than just grazing zero.


In [ ]:
# ---- PC attractor, non-custom (plateau-search) parameters ----
# ATTR_HP_PICK = PICKS['PC'] from section 4's det-sweep plateau search (cache/final_PC.npz) --
# the tied-plateau cell with the strongest retention contrast, NOT literal best_params('PC','small')
# (eta=1e-5, which freezes drive/reversal entirely -- see markdown above) and NOT the hand-picked
# HP_PC_CUSTOM. Loaded straight from the cache so it can never drift from what section 4/6 actually
# use; falls back to the same values hardcoded in case the cache hasn't been generated yet.
_pc_pick_path = Path('cache') / 'final_PC.npz'
if _pc_pick_path.exists():
    _pc_pick = np.load(_pc_pick_path, allow_pickle=True)
    ATTR_HP_PICK = dict(alpha=float(_pc_pick['alpha']), beta=float(_pc_pick['beta']),
                        eta=float(_pc_pick['eta']), tau=float(_pc_pick['tau']))
else:
    ATTR_HP_PICK = dict(alpha=0.037275937203149416, beta=0.0071968567300115215, eta=0.01, tau=0.0)
print('ATTR_HP_PICK (PC, plateau-search pick):', ATTR_HP_PICK)

TRAJ_SEED_PICK   = 44                            # found by scanning ATTR_N_SEEDS seeds for the 3-way dissociation
TRAJ_STEPS_PICK  = int(0.8 * ATTR_RETAIN_STEPS)  # trimmed from the full retention window -- drive and
                                                  # reversal have both crossed clearly negative by here;
                                                  # the last stretch just adds line clutter for no extra signal
TRAJ_STRIDE_PICK = 4 * TRAJ_STRIDE               # coarser than the custom-param plot's stride -- this
                                                  # trajectory is noisier step-to-step (real eta=0.01
                                                  # activity noise, not eta=0.003), so the same stride
                                                  # read as a muddy, over-plotted line; sampling it more
                                                  # sparsely keeps the overall path legible

# simulate the PC line attractor under all conditions (re-run after editing ATTR_HP_PICK / TRAJ_SEED_PICK)
attr_pick = {name: run_forgetting('PC', par, ATTR_HP_PICK, ATTR_TRAIN_STEPS, ATTR_RETAIN_STEPS, ATTR_N_SEEDS)
             for name, par in ATTR_COND.items()}
print('non-custom attractor sims done:', len(attr_pick), 'panels')


In [ ]:
from matplotlib.lines import Line2D
# ---- PC line attractor (non-custom / plateau-search params): same layout as the custom-param
# figure above, just a different operating point and TRAJ_SEED/TRAJ_STEPS (see markdown). The
# example trajectory is a plain thin line (not a time colormap) -- start/end markers already
# carry the direction, so the colormap was just visual noise on top of them. ----
rng2 = np.random.default_rng(0)
XL, YL = ATTR_LIM
names = list(ATTR_COND)
ref2 = attr_pick['trained']; rdw2 = memory(ref2)[-CLOUD_PTS:].ravel(); rdx2 = readout(ref2)[-CLOUD_PTS:].ravel()

fig, axes = plt.subplots(1, len(names), figsize=(1.55 * len(names), 1.75))
for ci, name in enumerate(names):
    ax = axes[ci]; st = attr_pick[name]; dw = memory(st); dx = readout(st)
    lo, hi = max(XL[0], YL[0]), min(XL[1], YL[1])
    ax.plot([lo, hi], [lo, hi], color='#ececec', lw=0.7, zorder=0)            # Δx = Δw guide
    if name != 'trained':                                                      # faint trained reference
        si = rng2.choice(rdw2.size, min(CLOUD_PTS, rdw2.size), replace=False)
        ax.scatter(rdw2[si], rdx2[si], s=2.5, c='#d6d6d6', lw=0, alpha=0.6, zorder=1, rasterized=True)
    cdw = dw[-CLOUD_PTS:].ravel(); cdx = dx[-CLOUD_PTS:].ravel()               # settled cloud
    sj = rng2.choice(cdw.size, min(CLOUD_PTS, cdw.size), replace=False)
    ax.scatter(cdw[sj], cdx[sj], s=4.5, c=ATTR_COLOR[name], lw=0, alpha=0.45, zorder=2, rasterized=True)
    if name != 'trained':                                                      # single example trajectory
        sub = slice(0, min(TRAJ_STEPS_PICK, dw.shape[0]), TRAJ_STRIDE_PICK)
        x = dw[sub, TRAJ_SEED_PICK]; y = dx[sub, TRAJ_SEED_PICK]
        ax.plot(x, y, color='#333333', lw=0.6, alpha=0.8, zorder=4)             # example trajectory
        ax.scatter([x[0]], [y[0]], s=24, facecolors='white', edgecolors='#222222',
                   lw=0.8, zorder=5)                                            # START
        ax.scatter([x[-1]], [y[-1]], marker='*', s=60, facecolors='white',
                   edgecolors='#222222', lw=0.6, zorder=6)                      # END
    ax.text(0.96, 0.06, f'$\Delta w$={dw[-CLOUD_PTS:].mean():+.2f}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=FS_S, color=ATTR_COLOR[name])
    ax.set_xlim(*XL); ax.set_ylim(*YL)
    nature_ax(ax, 'memory  $\Delta w$' if ci == len(names) // 2 else '',
              'activity  $\Delta x$' if ci == 0 else '', nx=3, ny=3)
    ax.set_title(ATTR_TITLE[name], fontsize=FS_L, color=ATTR_COLOR[name])
    if ci != 0: ax.tick_params(labelleft=False)
axes[-1].legend(handles=[Line2D([0], [0], marker='o', color='none', markerfacecolor='white',
                                markeredgecolor='#222', markersize=5, label='start'),
                         Line2D([0], [0], marker='*', color='none', markerfacecolor='white',
                                markeredgecolor='#222', markersize=8, label='end')],
                fontsize=FS_S, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                handletextpad=0.3, title='example', title_fontsize=FS_S)
fig.suptitle('PC line attractor, non-custom (plateau-search) params: memory Δw vs activity Δx   '
            '(grey = trained reference · line = example trajectory)',
             fontsize=FS_L, y=1.04)
fig.tight_layout()
if SAVE:
    fig.savefig(FIGDIR / 'fig_attractors_noncustom.svg', dpi=300, bbox_inches='tight')
plt.show()


## 8. Statistics: is retention higher with no input, and does reversal cross zero?

Two claims made informally above get a proper test here:

1. **No input retains memory significantly more than input-without-a-task.** Every seed's
   retention trajectory under both conditions starts from the *same* trained checkpoint (see
   `retain_full`'s callers in section 4) -- so per seed, "No input" and "Input, no new task" are
   **paired** observations, not two independent samples. The unit of analysis is one number
   *per seed* (`N_SEEDS` = 250 seeds per rule), never a per-timestep sample: within a trajectory,
   adjacent timesteps are highly autocorrelated, so treating `retain_steps x n_seeds` values as
   independent would massively inflate significance.

   Per seed, "retention" here is the **tail-mean of Δw over the last 20% of the retention
   window** (matches the `last_20pct` convention already used in section 5), not a single noisy
   last-step sample. Tested with a **paired, one-sided Wilcoxon signed-rank test** (primary -- no
   normality assumption, robust to the skewed/unstable rules flagged in sections 5-6) and a
   **paired one-sided t-test** (robustness check) for "No input" > "Input, no new task".

2. **Reversal has crossed to negative Δw by the end of the retention window.** Same per-seed
   tail-mean statistic, tested with a **one-sample, one-sided Wilcoxon signed-rank test** (primary)
   and a **one-sample one-sided t-test** (robustness check) against 0, alternative = "less than 0".

Both run for every rule in `ALL_RULES`, using each rule's picked cell from section 4
(`forgets_by_rule` -- the same full-fidelity data sections 5-7 already use), so every rule in the
paper's rule-wide screen gets a number, not just PC. With 7 rules x 2 questions = 14 tests,
p-values are also **Benjamini-Hochberg FDR-corrected** across all 14 so the rule-wide screen
doesn't rack up false positives from multiple comparisons.

Caveat worth keeping in mind when reading the table: for rules flagged in sections 5-6 as
oscillating/unstable rather than cleanly decaying (BCM, and to a lesser extent LC/HZ), a
"significant" result here says the *tail-mean* sits reliably on one side of the comparison --
it does not mean the trajectory is well-described as monotonic decay.


In [ ]:
import pandas as pd
from scipy import stats as sstats

def tail_mean_per_seed(arr, tail_frac=0.2):
    """Per-seed tail-mean over the last tail_frac of steps -- (n_seeds,).
    Keeps the seed axis (unlike section 5's tail_mean, which also averages
    over seeds); this is the correct per-seed statistic for a paired/
    one-sample test across seeds."""
    T = arr.shape[0]
    tail = max(1, int(T * tail_frac))
    return np.asarray(arr[-tail:]).mean(axis=0)

def rank_biserial(d):
    """Matched-pairs rank-biserial effect size for a signed-rank test on
    differences d (Kerby 2014 simple-difference formula): rank |d|, sum the
    ranks on the positive vs. negative side, r = (W+ - W-) / (W+ + W-).
    Exact zeros are dropped first, matching scipy.wilcoxon's default
    zero_method='wilcox'."""
    d = np.asarray(d); d = d[d != 0]
    if d.size == 0:
        return np.nan
    ranks = sstats.rankdata(np.abs(d))
    w_pos = ranks[d > 0].sum(); w_neg = ranks[d < 0].sum()
    return float((w_pos - w_neg) / (w_pos + w_neg))

def bh_fdr(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values (independent/positively-
    correlated tests assumption)."""
    p = np.asarray(pvals, dtype=float); n = p.size
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out = np.empty(n); out[order] = adj
    return out


In [ ]:
TAIL_FRAC_STATS = 0.2   # matches section 5's 'last_20pct' convention

stat_rows = []
for rule in ALL_RULES:
    forget = forgets_by_rule[rule]
    dw_noinput  = tail_mean_per_seed(memory(forget['No input']),           TAIL_FRAC_STATS)
    dw_notask   = tail_mean_per_seed(memory(forget['Input, no new task']), TAIL_FRAC_STATS)
    dw_reversal = tail_mean_per_seed(memory(forget['Reversal']),           TAIL_FRAC_STATS)

    # ---- Q1: No input > Input, no new task (paired across seeds -- same trained checkpoint) ----
    d1 = dw_noinput - dw_notask
    w1 = sstats.wilcoxon(d1, alternative='greater')
    t1 = sstats.ttest_rel(dw_noinput, dw_notask, alternative='greater')

    # ---- Q2: Reversal < 0 by the end of the window (one-sample against 0) ----
    w2 = sstats.wilcoxon(dw_reversal, alternative='less')
    t2 = sstats.ttest_1samp(dw_reversal, popmean=0., alternative='less')

    stat_rows.append(dict(
        rule=rule, n_seeds=dw_noinput.size,
        # Q1: no-input vs. input-no-task
        q1_median_diff=float(np.median(d1)), q1_mean_diff=float(np.mean(d1)),
        q1_wilcoxon_p=float(w1.pvalue), q1_wilcoxon_r=rank_biserial(d1),
        q1_ttest_p=float(t1.pvalue), q1_cohens_dz=float(np.mean(d1) / np.std(d1, ddof=1)),
        # Q2: reversal vs. 0
        q2_median=float(np.median(dw_reversal)), q2_mean=float(np.mean(dw_reversal)),
        q2_wilcoxon_p=float(w2.pvalue), q2_wilcoxon_r=rank_biserial(dw_reversal),
        q2_ttest_p=float(t2.pvalue),
        q2_cohens_dz=float(np.mean(dw_reversal) / np.std(dw_reversal, ddof=1)),
    ))

stats_table = pd.DataFrame(stat_rows).set_index('rule')

# ---- BH-FDR across all 14 tests together (7 rules x {Q1, Q2}), Wilcoxon and t-test separately ----
stats_table[['q1_wilcoxon_p_fdr', 'q2_wilcoxon_p_fdr']] = bh_fdr(
    stats_table[['q1_wilcoxon_p', 'q2_wilcoxon_p']].values.ravel()).reshape(-1, 2)
stats_table[['q1_ttest_p_fdr', 'q2_ttest_p_fdr']] = bh_fdr(
    stats_table[['q1_ttest_p', 'q2_ttest_p']].values.ravel()).reshape(-1, 2)

pd.set_option('display.max_columns', None); pd.set_option('display.width', 200)
stats_table.round(4)


In [ ]:
ALPHA_SIG = 0.05
print(f"{'rule':<5} | Q1: No input > Input, no-task (paired)                       "
     f"| Q2: Reversal < 0 (one-sample)")
print('-' * 118)
for rule, row in stats_table.iterrows():
    sig1 = 'YES' if row['q1_wilcoxon_p_fdr'] < ALPHA_SIG else 'no'
    sig2 = 'YES' if row['q2_wilcoxon_p_fdr'] < ALPHA_SIG else 'no'
    print(f"{rule:<5} | diff={row['q1_median_diff']:+.4f}  r={row['q1_wilcoxon_r']:+.2f}  "
         f"p_fdr={row['q1_wilcoxon_p_fdr']:.2e}  sig={sig1:<3} | "
         f"median={row['q2_median']:+.4f}  r={row['q2_wilcoxon_r']:+.2f}  "
         f"p_fdr={row['q2_wilcoxon_p_fdr']:.2e}  sig={sig2}")
print()
print("p-values are Wilcoxon signed-rank (primary), BH-FDR-corrected across all 7 rules x 2 "
     "questions (14 tests). r is the matched-pairs rank-biserial effect size (+1 = every seed "
     "agrees with the direction tested, 0 = no consistent direction). See stats_table for the "
     "full table incl. the paired/one-sample t-test run as a robustness check.")


### Figure-legend stats


> Paired one-sided Wilcoxon signed-rank test (no-input vs. constant-input) and one-sample
> one-sided Wilcoxon signed-rank test against 0 (reversal), tail-mean Δw over the last 20% of
> the retention window, n=250 seeds/rule, Benjamini-Hochberg FDR-corrected across 7 rules x 2
> tests. No-input retention was significantly higher than constant-input in BCM, OJA, PC, COV,
> HZ, HR (all p_FDR < 10⁻⁸) but not LC (p_FDR = 1.0, wrong-signed). Reversal reached
> significantly negative Δw in PC, COV, HR (all p_FDR < 10⁻⁹) but not BCM, OJA, LC, HZ
> (p_FDR = 1.0).

**Full per-rule table:**

| Rule | No-input vs. const.-input: r, p_FDR | Reversal vs. 0: r, p_FDR |
|---|---|---|
| BCM | r=+1.00, p_FDR<10⁻⁹ \*\*\* | r=+0.97, p_FDR=1.0 n.s. |
| OJA | r=+0.43, p_FDR=2.7×10⁻⁹ \*\*\* | r=+0.59, p_FDR=1.0 n.s. |
| LC  | r=−0.64, p_FDR=1.0 n.s. | r=+1.00, p_FDR=1.0 n.s. |
| PC  | r=+0.41, p_FDR=1.9×10⁻⁸ \*\*\* | r=−1.00, p_FDR<10⁻⁹ \*\*\* |
| COV | r=+0.98, p_FDR<10⁻⁹ \*\*\* | r=−1.00, p_FDR<10⁻⁹ \*\*\* |
| HZ  | r=+0.97, p_FDR<10⁻⁹ \*\*\* | r=+1.00, p_FDR=1.0 n.s. |
| HR  | r=+1.00, p_FDR<10⁻⁹ \*\*\* | r=−1.00, p_FDR<10⁻⁹ \*\*\* |

(r = matched-pairs rank-biserial effect size; \*\*\* p_FDR<0.001, n.s. not significant;
n=250 seeds/rule. Values above transcribed from `stats_table` two cells up, with p_FDR
capped at 10⁻⁹ for readability rather than reported to floating-point precision -- rerun
the cells above and re-copy this table if `CONFIG` or `ALL_RULES`/`PICKS` change.)
